# Week 5: Function Approximation

Craig Rudman  
Regis University  
MSDS684 Reinforcement Learning  
Prof. Mike Busch  

## Section 1: Project Overview 

This lab investigates function approximation, a method for representing value estimates that scales and generalizes better than the tabular methods used in our previous labs. Tabular methods store an estimate for each individual state or state-action pair, which works well when the state space is discrete and small. For continuous state spaces the number of possible states is infinite, making tabulation impossible. In addition, a tabular approach exactly maps states or state-action pairs to estimates, which makes generalization impossible.

Function approximation overcomes these constraints by discretizing the continuous state space into regions, and assigning weights to each region. Generalization is further aided by using multiple overlapping discretizations, slightly offset, creating a kind of anti-aliasing effect at the region boundaries. Estimates are therefore blended into a pseudo-continuous approximation. The tradeoff is resolution versus generalization: regions that are too large blur important distinctions, while regions that are too small reduce the sharing of experience between nearby states.

We will demonstrate the concept using the Gymnasium "MountainCar-v0" environment, where state is represented in two dimensions: position, with a continuous range (-1.2, 0.6), and velocity, with a continuous range (-0.07, 0.07). There are three discrete actions: push left, push right, and coast. The goal is reached when position is >= 0.5. Episodes truncate after 200 steps, or terminate when the goal is reached. The reward structure is simple: each step incurs a penalty of -1.

We will implement function approximation for value prediction using a method called "semi-gradient Sarsa," an on-policy approach described in Sutton and Barto, chapters 9 and 10, using ε-greedy exploration for action selection. Here, Sarsa, which is a one-step ahead temporal difference (TD) algorithm, is described as semi-gradient because the mechanism for estimation uses the same weights in the prediction and the target. Instead of deriving the learning gradient from stable values, we are using targets that change as we make predictions. This complicates finding the gradient, and so we don't factor this change into the calculation... we treat the target as stable and apply the update to the prediction. That's why they call it "semi-gradient."

A variety of approaches can be used to implement the value function; this demonstration uses a linear function for regression. The linear regression uses a weight vector that is tuned as the agent interacts with the environment. A binary feature vector derived from the state masks the weight vector, ensuring that the appropriate weights are updated for a given state.

The mechanism for feature construction is tile coding. This approach lays multiple overlapping grids, or "tilings", over the continuous state space, each one slightly offset. For a given state, each tiling identifies which tile the state falls into. The combination of active tiles across all tilings forms the feature vector.

We will experiment with the number of tilings, tile sizes, and offsets to determine their effects. I predict that there is an optimal configuration balancing resolution and generalization, and that deviations in either direction will hurt learning — too few or too coarse tilings will lack the resolution to distinguish important states, while too many or too fine tilings will reduce generalization and require more experience to learn effectively before episodes truncate.

## Section 2: Deliverables

In [ ]:
'''
Weight: 35 points (35%)

This section combines your implementation summary, results, and analysis.
'''

### GitHub Repository

In [ ]:
'''GitHub Repository URL (Required)
Place prominently at the top of this section:

GitHub Repository: https://github.com/username/repo/tree/main/lab#

Note: Your repository contains complete code, all experiments, and additional visualizations. 
This section highlights key findings with deep analysis.

What’s in GitHub Repository
All code (runnable, reproducible)
All generated visualizations
README with setup instructions
Additional experiments beyond what’s in PDF
requirements.txt or environment.yml
'''


### Implementation Summary

In [ ]:
'''Implementation Summary (100-150 words)

Brief prose description:

What you implemented (algorithms, environments)
Experimental setup (e.g., “1000 episodes, 30 random seeds, ε=0.1”)
Key hyperparameters chosen
NOT detailed pseudocode or line-by-line methods
'''


#### Key Results and Analysis

In [ ]:
'''Key Results & Analysis (400-600 words + 2-4 visualizations)
⚠️ CRITICAL RULES:

NO raw code listings in PDF
NO console output dumps
Code lives in GitHub; analysis lives here
Include 2-4 key visualizations:

Embedded directly in PDF (PNG/JPG format)
Each must have detailed interpretive caption
Choose your most important/insightful results
Captions must be interpretive, not just descriptive:

❌ Bad caption: “Figure 1 shows a line going up”

✅ Good caption: “Figure 1: UCB’s regret curve flattens after 500 steps because the
uncertainty bonus decreases as actions are sampled more, focusing exploration on 
genuinely uncertain actions (S&B Ch. 2.7). 
Final regret is 15% lower than ε-greedy (ε=0.1), demonstrating more efficient exploration.”

Discussion must address:

What do results show about algorithm behavior?
How do they relate to theory from Sutton & Barto? (cite chapters/sections)
What didn’t work as expected? Why?
How did hyperparameters affect performance?
What does this teach you about the RL concept?
Focus: Interpretation and understanding, not exhaustive description of every result.
'''

## Section 3: AI Use Reflection

### 3.1 Initial Interaction

I asked Claude to help me understand the conceptual foundations of semi-gradient Sarsa with tile coding for the MountainCar environment. Rather than asking for code, my initial prompts focused on understanding the algorithm: what "semi-gradient" means, how tile coding constructs features from continuous states, and how weights produce Q-value estimates. The AI provided explanations, corrected my misconceptions, and helped me edit the project overview section.

### 3.2 Iteration Cycle

#### 3.2.1 Iteration 1

In my first iteration, I worked with Claude to build a MountainCarEnvironment facade over Gymnasium using a test-first approach. My tests caught a bug immediately: I had swapped the observation_space and action_space properties in my implementation. Claude identified this from the red/green output and the fix was straightforward. We initially used a factory pattern to construct the environment, but decided it was unnecessary, opting for a singleton instead.

#### 3.3.2 Iteration 2

In my second iteration, I worked with Claude to implement the TileCoder class. We evaluated whether to expose both get_tiles and get_features, ultimately dropping get_features since the agent can index weight vectors directly using the active indices. We also reviewed the instructor's reference implementation critically, agreeing on two improvements: fixing mutable default arguments and replacing int(np.prod(...)) with math.prod.

### 3.3 Critical Evaluation

In [ ]:
'''
(50-75 words)

Did you catch any mistakes the AI made?
Did you test alternative approaches?
How did you verify the final solution was correct?
'''

### 3.4 Learning Reflection

In [ ]:
'''
(50-75 words)

What did you learn about the RL algorithm through debugging?
What did you learn about working with AI tools?
What would you do differently next time?
'''

In [ ]:
'''
Grading Criteria

Excellent (30-35 points):
    Documents 3+ specific debugging cycles
    Shows critical evaluation of AI suggestions
    Demonstrates learning from the process
    Specific, detailed examples

Good (24-29 points):
    Documents 2 debugging cycles
    Some critical thinking shown
    Generic learning statements
    Less specific

Adequate (18-23 points):
    Mentions iteration but lacks detail
    Minimal critical evaluation
    Vague descriptions

Poor (0-17 points):
    “I used AI to generate code”
    No iteration documented
    No evidence of critical thinking
    Appears to accept first draft
'''

## Section 4: Speaker Notes

In [ ]:
'''
(~5 minutes / 5-7 bullets)

Weight: 10 points (10%)
Purpose: Practice communicating technical work concisely. These could be used for peer presentations or your own reference.

Outline a brief presentation of your work:

Format: 5-7 bullet points covering:

Problem statement and motivation
Method and key algorithmic choices
Important design decision or challenge you faced
Main result or finding
Key insight or learning
(Optional) Connection to future weeks or real-world applications
Example:

• Problem: Exploration-exploitation in 10-armed bandits with non-stationary rewards
• Method: Implemented ε-greedy and UCB with constant step-size (α=0.1)
• Design choice: Used constant α instead of sample average to track changing rewards
• Key result: UCB outperformed ε-greedy after 500 steps (15% lower regret)
• Insight: Uncertainty-based exploration more efficient than random
• Challenge: Debugging action selection logic took 6 iterations with Claude
• Connection: This exploration concept extends to full MDPs in Week 2
'''